# Evolving research themes in animal science and veterinary journals using text mining and BERTopic

This notebook accompanies the manuscript **"Evolving research themes in animal science and veterinary journals using text mining and BERTopic"**.

## How to run

1. Place the Scopus CSV export in the `data/` folder. The notebook accepts `.csv` or `.csv.gz` files.
2. Run the cells from top to bottom.
3. Generated figures and tables can be saved to the `results/` folder if needed.

## Notes

- The notebook uses relative paths for GitHub compatibility.
- Cell outputs were cleared for public release.
- Check Scopus/Elsevier terms before redistributing the full raw CSV export.


# Cell 1. 패키지 불러오기

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

# Cell 2. 작업 폴더와 CSV 파일 확인

In [ ]:
# GitHub-friendly project paths
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR = PROJECT_ROOT / "models"

DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

# The notebook accepts either .csv or .csv.gz files.
csv_files = sorted(list(DATA_DIR.glob("*.csv")) + list(DATA_DIR.glob("*.csv.gz")))

# Fallback: also check the repository root.
if len(csv_files) == 0:
    csv_files = sorted(list(PROJECT_ROOT.glob("*.csv")) + list(PROJECT_ROOT.glob("*.csv.gz")))

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV file was found. Place the Scopus export file in the 'data/' folder."
    )

print("CSV files found:")
for i, f in enumerate(csv_files, 1):
    print(f"{i}. {f.name}")

# Cell 3. Scopus CSV 로드

In [ ]:
csv_path = csv_files[0]

df_raw = pd.read_csv(
    csv_path,
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

print("Loaded file:", csv_path.name)
print("Shape:", df_raw.shape)

display(df_raw.head())

# Cell 4. 컬럼명 확인

In [ ]:
print("Columns:")
for col in df_raw.columns:
    print("-", col)

# Cell 5. 분석용 데이터 복사 및 기본 형식 정리

In [ ]:
df = df_raw.copy()

# Year, Cited by 숫자형 변환
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Cited by"] = pd.to_numeric(df["Cited by"], errors="coerce").fillna(0)

# Source title, Document Type 정리
df["Source title"] = (
    df["Source title"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df["Document Type"] = (
    df["Document Type"]
    .astype(str)
    .str.strip()
    .str.title()
)

# ISSN 문자열 정리
df["ISSN"] = (
    df["ISSN"]
    .astype(str)
    .str.strip()
    .replace({"nan": "", "None": ""})
)

print("Shape after basic formatting:", df.shape)
display(df[["Title", "Year", "Source title", "Document Type", "ISSN"]].head())

# Cell 6. 저널명 원본 표기 확인

In [ ]:
source_check = (
    df["Source title"]
    .value_counts()
    .reset_index()
)

source_check.columns = ["Source title", "Count"]

display(source_check)

# Cell 7. 저널명 정규화

In [ ]:
journal_map = {
    "Animal": "Animal",
    "Animal Nutrition": "Animal Nutrition",
    "Journal of Animal Science and Biotechnology": "Journal of Animal Science and Biotechnology",
    "Journal of Animal Science and Technology": "Journal of Animal Science and Technology",
    "Pakistan Veterinary Journal": "Pakistan Veterinary Journal",
    "Veterinary Research": "Veterinary Research",
    "Veterinary research": "Veterinary Research",
    "Lab Animal": "Lab Animal",
    "Lab animal": "Lab Animal",
    "International Journal of Veterinary Science and Medicine": "International Journal of Veterinary Science and Medicine",
    "International Journal of Veterinary Science and Medicine ": "International Journal of Veterinary Science and Medicine"
}

df["Journal"] = df["Source title"].replace(journal_map)

journal_check = (
    df["Journal"]
    .value_counts()
    .reset_index()
)

journal_check.columns = ["Journal", "Count"]

display(journal_check)

# Cell 8. 분석 대상 저널과 문헌 유형 필터링

In [ ]:
selected_journals = [
    "Animal Nutrition",
    "Journal of Animal Science and Biotechnology",
    "Pakistan Veterinary Journal",
    "Animal",
    "Lab Animal",
    "Veterinary Research",
    "International Journal of Veterinary Science and Medicine",
    "Journal of Animal Science and Technology"
]

df_main = df[
    (df["Journal"].isin(selected_journals)) &
    (df["Document Type"].isin(["Article", "Review"])) &
    (df["Year"].between(2010, 2025))
].copy()

print("Raw data shape:", df.shape)
print("Filtered data shape:", df_main.shape)

display(
    df_main.groupby(["Journal", "Document Type"])
    .size()
    .unstack(fill_value=0)
)

# Cell 9. DOI 기준 중복 확인

In [ ]:
doi_nonempty = df_main["DOI"].notna() & (df_main["DOI"].astype(str).str.strip() != "")

n_total = len(df_main)
n_doi = doi_nonempty.sum()
n_duplicated_doi = df_main.loc[doi_nonempty, "DOI"].duplicated().sum()

print("Total records:", n_total)
print("Records with DOI:", n_doi)
print("Duplicated DOI records:", n_duplicated_doi)

dup_doi_table = (
    df_main.loc[doi_nonempty & df_main["DOI"].duplicated(keep=False),
                ["DOI", "Title", "Year", "Journal", "Document Type"]]
    .sort_values("DOI")
)

display(dup_doi_table.head(30))

# Cell 10. DOI + Title 기준 중복 제거

In [ ]:
df_clean = df_main.copy()

# DOI가 있는 경우 DOI 기준 중복 제거
df_clean = df_clean.sort_values(["DOI", "Title", "Year"])
df_clean = df_clean.drop_duplicates(subset=["DOI"], keep="first")

# DOI가 비어 있는 경우를 대비해 Title + Year + Journal 기준 추가 중복 제거
df_clean = df_clean.drop_duplicates(subset=["Title", "Year", "Journal"], keep="first")

print("Before duplicate removal:", len(df_main))
print("After duplicate removal:", len(df_clean))

# Cell 11. Table 1 생성

In [ ]:
def collect_issn(x):
    values = []
    for item in x.dropna().astype(str):
        item = item.strip()
        if item == "" or item.lower() == "nan":
            continue
        
        # Scopus ISSN이 여러 개일 경우 separator가 다양할 수 있어 처리
        parts = re.split(r"[;,]\s*|\s{2,}", item)
        for p in parts:
            p = p.strip()
            if p and p.lower() != "nan":
                values.append(p)
    
    # 중복 제거, 순서 유지
    unique_values = list(dict.fromkeys(values))
    return "; ".join(unique_values)

table1 = (
    df_clean
    .groupby("Journal")
    .agg(
        ISSN_eISSN=("ISSN", collect_issn),
        Start_year=("Year", "min"),
        End_year=("Year", "max"),
        Article=("Document Type", lambda x: (x == "Article").sum()),
        Review=("Document Type", lambda x: (x == "Review").sum()),
        Total_documents=("Document Type", "count")
    )
    .reset_index()
)

table1["Analysis period"] = (
    table1["Start_year"].astype(int).astype(str) + "–" +
    table1["End_year"].astype(int).astype(str)
)

table1 = table1[
    ["Journal", "ISSN_eISSN", "Analysis period", "Article", "Review", "Total_documents"]
]

# 지정한 저널 순서대로 정렬
table1["Journal"] = pd.Categorical(table1["Journal"], categories=selected_journals, ordered=True)
table1 = table1.sort_values("Journal").reset_index(drop=True)

# Total row 추가
total_row = pd.DataFrame({
    "Journal": ["Total"],
    "ISSN_eISSN": [""],
    "Analysis period": [
        f"{int(df_clean['Year'].min())}–{int(df_clean['Year'].max())}"
    ],
    "Article": [int((df_clean["Document Type"] == "Article").sum())],
    "Review": [int((df_clean["Document Type"] == "Review").sum())],
    "Total_documents": [int(len(df_clean))]
})

table1_final = pd.concat([table1, total_row], ignore_index=True)

display(table1_final)

# Cell 12. Table 1용 caption 확인

In [ ]:
table1_caption = (
    "Table 1. Overview of the selected animal science and veterinary journals, "
    "analysis periods, and number of articles and reviews included in the text-mining dataset."
)

print(table1_caption)
display(table1_final)

# Cell 13. Figure 2용 연도별 데이터 만들기

In [ ]:
# Figure 2용 연도별 publication / citation summary

year_range = range(2010, 2026)

annual_trend = (
    df_clean
    .groupby("Year")
    .agg(
        Total_documents=("Title", "count"),
        Articles=("Document Type", lambda x: (x == "Article").sum()),
        Reviews=("Document Type", lambda x: (x == "Review").sum()),
        Total_citations=("Cited by", "sum"),
        Mean_citations=("Cited by", "mean"),
        Median_citations=("Cited by", "median")
    )
    .reindex(year_range)
    .fillna(0)
    .reset_index()
)

annual_trend["Year"] = annual_trend["Year"].astype(int)
annual_trend[["Total_documents", "Articles", "Reviews", "Total_citations"]] = (
    annual_trend[["Total_documents", "Articles", "Reviews", "Total_citations"]]
    .astype(int)
)

display(annual_trend)

# Cell 14. 전체 publication/citation 요약 확인

In [ ]:
print("Analysis period:", annual_trend["Year"].min(), "–", annual_trend["Year"].max())
print("Total documents:", annual_trend["Total_documents"].sum())
print("Total articles:", annual_trend["Articles"].sum())
print("Total reviews:", annual_trend["Reviews"].sum())
print("Total citations:", annual_trend["Total_citations"].sum())

peak_pub_year = annual_trend.loc[annual_trend["Total_documents"].idxmax(), "Year"]
peak_pub_count = annual_trend["Total_documents"].max()

peak_cit_year = annual_trend.loc[annual_trend["Total_citations"].idxmax(), "Year"]
peak_cit_count = annual_trend["Total_citations"].max()

print(f"Peak publication year: {peak_pub_year} ({peak_pub_count} documents)")
print(f"Peak citation year: {peak_cit_year} ({peak_cit_count} citations)")

# Cell 15. Figure 2 그리기: annual publication count + citation count

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator

# Figure style
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.linewidth"] = 1
mpl.rcParams["xtick.major.width"] = 1
mpl.rcParams["ytick.major.width"] = 1
mpl.rcParams["xtick.direction"] = "out"
mpl.rcParams["ytick.direction"] = "out"

fig, ax1 = plt.subplots(figsize=(5, 4))

bar_width = 0.65
years = annual_trend["Year"].values

# Bar plot: annual document count
ax1.bar(
    years,
    annual_trend["Total_documents"],
    width=bar_width,
    label="Documents",
    color="#E4AAD9",
    align="center"
)
#BBBBBB
ax1.set_xlabel("Publication year", fontsize=13)
ax1.set_ylabel("Number of documents", fontsize=13)

# x-axis ticks exactly at the center of each bar
ax1.set_xticks(years)
ax1.set_xticklabels(years, rotation=45, ha="right")

# Reduce left and right margins of x-axis
x_min = annual_trend["Year"].min()
x_max = annual_trend["Year"].max()
ax1.set_xlim(x_min - 0.55, x_max + 0.55)

ax1.tick_params(axis="both", labelsize=11)

# y-axis ticks and limit: exactly match top tick
locator1 = MaxNLocator(nbins=6, integer=True)
ticks1 = locator1.tick_values(0, annual_trend["Total_documents"].max())
ax1.set_yticks(ticks1)
ax1.set_ylim(0, ticks1[-1])

# Second axis: citation count
ax2 = ax1.twinx()

ax2.plot(
    years,
    annual_trend["Total_citations"],
    marker="o",
    markersize=7,
    markerfacecolor="none",
    markeredgewidth=1.25,
    linewidth=1.5,
    label="Citations",
    color="#6D6D6D"
)
#E4AAD9
ax2.set_ylabel("Citation count", fontsize=13)
ax2.tick_params(axis="y", labelsize=11)

# right y-axis ticks and limit: exactly match top tick
locator2 = MaxNLocator(nbins=6, integer=True)
ticks2 = locator2.tick_values(0, annual_trend["Total_citations"].max())
ax2.set_yticks(ticks2)
ax2.set_ylim(0, ticks2[-1])

# Full box style
for ax in [ax1, ax2]:
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)

# Legend from both axes
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    frameon=False,
    fontsize=12,
    loc="upper left"
)

plt.tight_layout()
plt.show()

# Cell 16. Figure 3용 패키지와 affiliation 컬럼 확인

In [ ]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

import geopandas as gpd

# Figure style
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.linewidth"] = 1.2
mpl.rcParams["xtick.major.width"] = 1.2
mpl.rcParams["ytick.major.width"] = 1.2
mpl.rcParams["xtick.direction"] = "out"
mpl.rcParams["ytick.direction"] = "out"

# Affiliation-related columns
affiliation_cols = [col for col in df_clean.columns if "affil" in col.lower() or "country" in col.lower()]
print("Affiliation-related columns:")
for col in affiliation_cols:
    print("-", col)

display(df_clean[["Title", "Journal", "Year", "Affiliations"]].head(5))

# Cell 17. 세계지도 데이터 로드

In [ ]:
# Load world map
# First try the old geopandas built-in dataset.
# If it is not available, read Natural Earth directly from URL.

try:
    world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
except Exception:
    url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
    world = gpd.read_file(url)

# Harmonize country name column
if "name" in world.columns:
    world = world.rename(columns={"name": "map_country"})
elif "ADMIN" in world.columns:
    world = world.rename(columns={"ADMIN": "map_country"})
elif "NAME" in world.columns:
    world = world.rename(columns={"NAME": "map_country"})
else:
    raise ValueError("Country name column was not found in the world map dataset.")

world = world[world["map_country"] != "Antarctica"].copy()

print(world.shape)
display(world[["map_country", "geometry"]].head())

# Cell 18. Affiliation에서 국가 추출

In [ ]:
country_corrections = {
    "USA": "United States",
    "U.S.A": "United States",
    "U.S.A.": "United States",
    "US": "United States",
    "United States of America": "United States",
    "UK": "United Kingdom",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Wales": "United Kingdom",
    "Northern Ireland": "United Kingdom",
    "Republic of Korea": "South Korea",
    "Korea": "South Korea",
    "Korea Republic": "South Korea",
    "Peoples R China": "China",
    "People's Republic of China": "China",
    "P.R. China": "China",
    "PR China": "China",
    "Russian Federation": "Russia",
    "Türkiye": "Turkey",
    "Viet Nam": "Vietnam",
    "Iran": "Iran",
    "Islamic Republic of Iran": "Iran",
    "Czech Republic": "Czechia",
    "UAE": "United Arab Emirates",
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Brunei Darussalam": "Brunei",
}

def clean_country_name(country):
    if pd.isna(country):
        return None
    
    country = str(country).strip()
    country = re.sub(r"\.$", "", country)
    country = re.sub(r"\s+", " ", country)
    
    # remove postal-code-like fragments at the end if accidentally included
    country = country.strip(" .;")
    
    if country in country_corrections:
        country = country_corrections[country]
    
    return country if country else None


def extract_countries_from_affiliations(affiliation_text):
    if pd.isna(affiliation_text) or str(affiliation_text).strip() == "":
        return []
    
    affiliation_text = str(affiliation_text)
    
    # Scopus affiliations are commonly separated by semicolons
    affiliations = re.split(r";\s*", affiliation_text)
    
    countries = []
    
    for aff in affiliations:
        aff = aff.strip()
        if not aff:
            continue
        
        parts = [p.strip() for p in aff.split(",") if p.strip()]
        if len(parts) == 0:
            continue
        
        # In Scopus affiliation strings, country is usually the last comma-separated element
        country = clean_country_name(parts[-1])
        
        if country is not None:
            countries.append(country)
    
    # Remove duplicates in the same article while preserving order
    countries = list(dict.fromkeys(countries))
    return countries


df_clean["Country_list"] = df_clean["Affiliations"].apply(extract_countries_from_affiliations)
df_clean["Number_of_countries"] = df_clean["Country_list"].apply(len)

display(
    df_clean[["Title", "Journal", "Year", "Affiliations", "Country_list", "Number_of_countries"]]
    .head(10)
)

# Cell 19. 국가 추출 결과 확인

In [ ]:
n_total_docs = len(df_clean)
n_with_country = (df_clean["Number_of_countries"] > 0).sum()
n_without_country = (df_clean["Number_of_countries"] == 0).sum()

print("Total documents:", n_total_docs)
print("Documents with extracted country:", n_with_country)
print("Documents without extracted country:", n_without_country)
print("Country extraction rate (%):", round(n_with_country / n_total_docs * 100, 2))

# Country frequency preview
all_countries = df_clean["Country_list"].explode()
country_freq_preview = all_countries.value_counts().reset_index()
country_freq_preview.columns = ["Country", "Frequency"]

display(country_freq_preview.head(40))

# Check documents without extracted country
display(
    df_clean.loc[df_clean["Number_of_countries"] == 0,
                ["Title", "Journal", "Year", "Affiliations"]]
    .head(10)
)

# Cell 20. 국가별 publication count, contribution score, 공동연구 유형 계산

In [ ]:
country_records = []

for idx, row in df_clean.iterrows():
    countries = row["Country_list"]
    
    if len(countries) == 0:
        continue
    
    n_countries = len(countries)
    contribution = 1 / n_countries
    collaboration_type = "Single-country" if n_countries == 1 else "International joint"
    
    for country in countries:
        country_records.append({
            "Record_ID": idx,
            "Year": row["Year"],
            "Journal": row["Journal"],
            "Country": country,
            "Publication_count": 1,
            "Contribution_score": contribution,
            "Collaboration_type": collaboration_type
        })

country_df = pd.DataFrame(country_records)

country_summary = (
    country_df
    .groupby("Country")
    .agg(
        Publication_count=("Publication_count", "sum"),
        Contribution_score=("Contribution_score", "sum"),
        Single_country=("Collaboration_type", lambda x: (x == "Single-country").sum()),
        International_joint=("Collaboration_type", lambda x: (x == "International joint").sum())
    )
    .reset_index()
)

country_summary["Total_collaboration_records"] = (
    country_summary["Single_country"] + country_summary["International_joint"]
)

country_summary["International_collaboration_ratio"] = (
    country_summary["International_joint"] / country_summary["Total_collaboration_records"]
)

country_summary["Normalized_contribution_score"] = (
    country_summary["Contribution_score"] / country_summary["Contribution_score"].max()
)

country_summary = (
    country_summary
    .sort_values("Publication_count", ascending=False)
    .reset_index(drop=True)
)

display(country_summary.head(30))

# Cell 21. 지도 데이터와 국가 분석 결과 병합

In [ ]:
# Match country names between Scopus-derived country names and Natural Earth map names

map_name_corrections = {
    "United States": "United States of America",
    "South Korea": "South Korea",
    "North Korea": "North Korea",
    "Russia": "Russia",
    "Turkey": "Turkey",
    "Czechia": "Czechia",
    "Vietnam": "Vietnam",
    "United Kingdom": "United Kingdom",
    "United Arab Emirates": "United Arab Emirates",
    "Bosnia and Herzegovina": "Bosnia and Herzegovina",
    "Brunei": "Brunei",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Democratic Republic of Congo": "Democratic Republic of the Congo",
    "Republic of Congo": "Republic of the Congo",
    "Tanzania": "United Republic of Tanzania",
    "Laos": "Laos",
    "Syria": "Syria",
    "Iran": "Iran",
    "Moldova": "Moldova",
    "Bolivia": "Bolivia",
    "Venezuela": "Venezuela",
}

map_df = country_summary.copy()
map_df["map_country"] = map_df["Country"].replace(map_name_corrections)

world_map = world.merge(
    map_df,
    how="left",
    on="map_country"
)

# Check unmatched countries
world_country_set = set(world["map_country"].dropna())
map_country_set = set(map_df["map_country"].dropna())

unmatched_countries = sorted(list(map_country_set - world_country_set))

print("Number of countries in country_summary:", country_summary["Country"].nunique())
print("Number of unmatched countries for map:", len(unmatched_countries))
print(unmatched_countries[:50])

display(
    world_map[["map_country", "Publication_count", "Contribution_score",
            "Normalized_contribution_score", "International_collaboration_ratio"]]
    .dropna(subset=["Publication_count"])
    .sort_values("Publication_count", ascending=False)
    .head(20)
)

# Cell 22. Figure 3 통합형: bar plot + inset world map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

# =========================
# Global style
# =========================
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.linewidth"] = 1
mpl.rcParams["xtick.major.width"] = 1
mpl.rcParams["ytick.major.width"] = 1
mpl.rcParams["xtick.minor.width"] = 1
mpl.rcParams["ytick.minor.width"] = 1
mpl.rcParams["xtick.direction"] = "out"
mpl.rcParams["ytick.direction"] = "out"

top_n = 25

# =========================
# Data preparation
# =========================
plot_a = country_summary.head(top_n).copy()
plot_a = plot_a.sort_values("Publication_count", ascending=False).reset_index(drop=True)

plot_b = country_summary.head(top_n).copy()
plot_b = plot_b.sort_values("Total_collaboration_records", ascending=False).reset_index(drop=True)

# =========================
# Custom colormaps
# =========================
cmap_a = LinearSegmentedColormap.from_list(
    "gray_to_skyblue",
    ["#D4D4D4BB", "#7EB2EA"]
)

cmap_b = LinearSegmentedColormap.from_list(
    "salmon_to_lightgreen",
    ["#F9D8B5", "#96E48E"]
)

# =========================
# Figure
# =========================
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Shared inset positions
# [left, bottom, width, height] in parent axes coordinates
map_pos = [0.20, 0.50, 0.72, 0.34]
cbar_pos = [0.49, 0.85, 0.28, 0.025]
cbar_label_y = 0.875

# World map bounds
xmin, ymin, xmax, ymax = world_map.total_bounds

# =========================
# Panel A
# =========================
ax = axes[0]
ax.set_zorder(2)
ax.patch.set_alpha(0)

# -------------------------------------------------
# Manual controls
# B panel과 같은 방식으로 조정
# -------------------------------------------------
map_pos_a = [0.01, 0.00, 0.99, 1.15]

# label과 colorbar를 수평으로 배치
# label_x_a: 글자 위치
# cbar_pos_a: 컬러바 위치
label_x_a = 0.52
label_y_a = 0.945
cbar_pos_a = [0.54, 0.935, 0.20, 0.018]

# -------------------------------------------------
# Map behind bars
# -------------------------------------------------
axins = ax.inset_axes(map_pos_a, zorder=0)

world_map.plot(
    column="Normalized_contribution_score",
    cmap=cmap_a,
    vmin=0,
    vmax=1,
    linewidth=0.3,
    edgecolor="lightgray",
    missing_kwds={"color": "#F0F0F0"},
    ax=axins
)

axins.set_xlim(-170, 170)
axins.set_ylim(-58, 85)
axins.set_aspect("equal", adjustable="box")
axins.margins(0)
axins.set_axis_off()

# -------------------------------------------------
# Colorbar: label and bar horizontally aligned
# -------------------------------------------------
ax.text(
    label_x_a,
    label_y_a,
    "Normalized contribution score",
    transform=ax.transAxes,
    ha="right",
    va="center",
    fontsize=8
)

cax = ax.inset_axes(cbar_pos_a, zorder=10)
cax.patch.set_alpha(0)

sm = ScalarMappable(norm=Normalize(vmin=0, vmax=1), cmap=cmap_a)
sm.set_array([])

cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
cb.set_ticks([0, 1])
cb.ax.tick_params(labelsize=7, length=2, pad=1, width=0.3)
cb.outline.set_linewidth(0.3)

# -------------------------------------------------
# Bars
# -------------------------------------------------
x = np.arange(len(plot_a))
bar_width = 0.45

ax.bar(
    x - bar_width/2,
    plot_a["Publication_count"],
    width=bar_width,
    color="#BBBBBB",
    label="Publication count",
    zorder=5
)

ax.bar(
    x + bar_width/2,
    plot_a["Contribution_score"],
    width=bar_width,
    color="#7EB2EA",
    label="Contribution score",
    zorder=5
)

ax.set_ylabel("Publication count & contribution score", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(plot_a["Country"], rotation=90, fontsize=9)
ax.tick_params(axis="y", labelsize=10, width=1)
ax.tick_params(axis="x", labelsize=9, width=1)

locator = MaxNLocator(nbins=6, integer=True)
ticks = locator.tick_values(
    0,
    max(plot_a["Publication_count"].max(), plot_a["Contribution_score"].max())
)
ax.set_yticks(ticks)
ax.set_ylim(0, ticks[-1])
ax.set_xlim(-0.6, len(plot_a) - 0.4)

ax.legend(
    frameon=False,
    fontsize=10,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.12),
    borderaxespad=0.0,
    ncol=2,
    columnspacing=1.2,
    handlelength=1.6
)

# =========================
# Panel B
# =========================
ax = axes[1]
ax.set_zorder(2)
ax.patch.set_alpha(0)

# -------------------------------------------------
# Manual controls
# 지도 위치는 현재 마음에 든 설정 유지
# -------------------------------------------------
map_pos_b = [0.01, 0.00, 0.99, 1.15]

# label과 colorbar를 수평으로 배치
label_x_b = 0.52
label_y_b = 0.945
cbar_pos_b = [0.54, 0.935, 0.20, 0.018]

# -------------------------------------------------
# Map behind bars
# -------------------------------------------------
axins = ax.inset_axes(map_pos_b, zorder=0)

world_map.plot(
    column="International_collaboration_ratio",
    cmap=cmap_b,
    vmin=0,
    vmax=1,
    linewidth=0.3,
    edgecolor="lightgray",
    missing_kwds={"color": "#F0F0F0"},
    ax=axins
)

axins.set_xlim(-170, 170)
axins.set_ylim(-58, 85)
axins.set_aspect("equal", adjustable="box")
axins.margins(0)
axins.set_axis_off()

# -------------------------------------------------
# Colorbar: label and bar horizontally aligned
# -------------------------------------------------
ax.text(
    label_x_b,
    label_y_b,
    "Int'l collaboration ratio",
    transform=ax.transAxes,
    ha="right",
    va="center",
    fontsize=8
)

cax = ax.inset_axes(cbar_pos_b, zorder=10)
cax.patch.set_alpha(0)

sm = ScalarMappable(norm=Normalize(vmin=0, vmax=1), cmap=cmap_b)
sm.set_array([])

cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
cb.set_ticks([0, 1])
cb.ax.tick_params(labelsize=7, length=2, pad=1, width=0.3)
cb.outline.set_linewidth(0.3)

# -------------------------------------------------
# Stacked bars
# -------------------------------------------------
x = np.arange(len(plot_b))

ax.bar(
    x,
    plot_b["Single_country"],
    width=0.75,
    color="#96E48E",
    label="Single-country research",
    zorder=5
)

ax.bar(
    x,
    plot_b["International_joint"],
    bottom=plot_b["Single_country"],
    width=0.75,
    color="#F9D8B5",
    label="International joint research",
    zorder=5
)

ax.set_ylabel("Single-country & international joint research count", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(plot_b["Country"], rotation=90, fontsize=9)
ax.tick_params(axis="y", labelsize=10, width=1)
ax.tick_params(axis="x", labelsize=9, width=1)

locator = MaxNLocator(nbins=6, integer=True)
ticks = locator.tick_values(0, plot_b["Total_collaboration_records"].max())
ax.set_yticks(ticks)
ax.set_ylim(0, ticks[-1])
ax.set_xlim(-0.6, len(plot_b) - 0.4)

ax.legend(
    frameon=False,
    fontsize=10,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.12),
    borderaxespad=0.0,
    ncol=2,
    columnspacing=1.2,
    handlelength=1.6
)

# =========================
# Full box style
# =========================
for ax in axes:
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)

plt.tight_layout()
plt.show()

# Cell 23. Keyword 관련 컬럼 확인

In [ ]:
# =========================
# Cell 23. Check keyword-related columns
# =========================

keyword_cols = [col for col in df_clean.columns if "keyword" in col.lower()]
print("Keyword-related columns:")
for col in keyword_cols:
    print("-", col)

display(df_clean[["Title", "Year", "Journal"] + keyword_cols].head(10))

# Cell 24. Keyword 전처리 함수 설정

In [ ]:
# =========================
# Cell 24. Keyword preprocessing functions - final revised version
# =========================

author_kw_col = "Author Keywords"
index_kw_col = "Index Keywords"

for col in [author_kw_col, index_kw_col]:
    if col not in df_clean.columns:
        print(f"Warning: {col} was not found.")

# General indexing terms and non-informative terms to remove
keyword_stopwords = {
    "",
    "nan",
    "none",
    
    # general publication / research terms
    "article",
    "review",
    "study",
    "studies",
    "research",
    "result",
    "results",
    "method",
    "methods",
    "methodology",
    "procedure",
    "procedures",
    "analysis",
    "data",
    "experiment",
    "experiments",
    "controlled study",
    "comparative study",
    "case report",
    "clinical article",
    "note",
    "time",
    "time factors",
    "classification",
    
    # overly broad biological / indexing terms
    "animal",
    "animals",
    "animalia",
    "nonhuman",
    "human",
    "humans",
    "male",
    "female",
    "adult",
    "aged",
    "young adult",
    "juvenile",
    "newborn",
    "infant",
    "child",
    
    # broad discipline / indexing terms
    "veterinary",
    "veterinary medicine",
    "science",
    "sciences",
    "journal",
    "journals",
    "paper",
    "papers",
    "field",
    "fields",
    
    # broad biological functions that are too generic alone
    "physiology",
    "metabolism",
    "genetics",
    "immunology",
    "microbiology",
    "chemistry",
    "virology",
    "growth development and aging",
    
    # generic Scopus / MeSH-like terms
    "animal experiment",
    "animal model",
    "animal models",
    "models biological",
    "animal tissue",
    "animal tissues",
    "animal disease",
    "animal diseases",
    "animal husbandry",
    "animal cell",
    "animal cells",
    "animal nutritional physiological phenomena",
    "biological model",
    
    # generic or technical indexing terms
    "unclassified drug",
    "standard",
    "standards",
    "drug effect",
    "drug effects",
    "blood",
    "pathology",
    "phenotype",
    "polymerase chain reaction",
    "real time polymerase chain reaction",
    "real-time polymerase chain reaction",
    "enzyme linked immunosorbent assay",
    "enzyme-linked immunosorbent assay",
    "protein expression",
    "gene expression regulation",
    "molecular sequence data",
    "upregulation",
    "bacteria microorganisms",
    "random allocation",
    "randomization",
    "isolation and purification",
    "dna extraction",
    "enzyme activity",
    "pathogenicity",
    "feces",
    
    # housing-related indexing terms
    "housing animal",
    "animal housing",
    
    # overly broad nutrition/feed terms for Table 2
    "animal food",
    "animal feed",
    "diet",
    "digestion",
    
    # species-level terms that dominate frequency table
    "bovine",
    "bos",
    "bovinae",
    "ovis aries",
    "suidae",
    "sus scrofa",
    "cattle",
    "pig",
    "swine",
    "mouse",
    "mice",
    "sheep",
    "chicken",
    "chickens",
    "broiler",
    "aves",
    "bovidae",
    
    # optional broad measurement terms
    "body weight"
}

# Synonym and spelling normalization
keyword_synonym_map = {
    # microbiome-related
    "microbiota": "microbiome",
    "gut microbiota": "microbiome",
    "intestinal microbiota": "microbiome",
    "intestinal microbiome": "microbiome",
    "gut microbiome": "microbiome",
    "intestine flora": "microbiome",
    "intestinal flora": "microbiome",
    "gut flora": "microbiome",
    "cecal microbiota": "cecal microbiome",
    "caecal microbiota": "cecal microbiome",
    "ruminal microbiota": "rumen microbiome",
    "rumen microbiota": "rumen microbiome",
    "ruminal microbiome": "rumen microbiome",
    
    # antimicrobial resistance
    "amr": "antimicrobial resistance",
    "antibiotic resistance": "antimicrobial resistance",
    "antimicrobial resistant": "antimicrobial resistance",
    
    # environmental terms
    "greenhouse gases": "greenhouse gas",
    "greenhouse gas emissions": "greenhouse gas",
    "ghg": "greenhouse gas",
    "methane emission": "methane emissions",
    "methane emissions": "methane emissions",
    "methane production": "methane emissions",
    
    # nutrition / feed
    "feed conversion ratio": "feed conversion ratio",
    "fcr": "feed conversion ratio",
    "feed efficiency": "feed efficiency",
    "growth performances": "growth performance",
    "growth performance": "growth performance",
    "dietary supplementation": "diet supplementation",
    "dietary supplement": "diet supplementation",
    "dietary supplements": "diet supplementation",
    "feed additives": "feed additive",
    "feed additive": "feed additive",
    "probiotics": "probiotic",
    "prebiotics": "prebiotic",
    
    # species normalization - only useful if species terms are later retained
    "broilers": "broiler",
    "broiler chickens": "broiler",
    "laying hens": "laying hen",
    "dairy cows": "dairy cow",
    "dairy cattle": "dairy cattle",
    "beef cattle": "beef cattle",
    "pigs": "pig",
    "piglets": "piglet",
    
    # pathogens
    "escherichia coli": "e. coli",
    "e coli": "e. coli",
    "e. coli": "e. coli",
    "salmonella spp": "salmonella",
    "salmonella spp.": "salmonella",
    
    # technology / concepts
    "machine-learning": "machine learning",
    "machine learning": "machine learning",
    "deep-learning": "deep learning",
    "deep learning": "deep learning",
    "precision livestock farming": "precision livestock farming",
    "one health": "one health",
    
    # welfare
    "welfare": "animal welfare",
    "animal welfare": "animal welfare",
    
    # products and production
    "meat quality": "meat quality",
    "milk quality": "milk quality",
    "milk": "milk",
    "meat": "meat",
    "lactation": "lactation",
    "dairying": "dairying",
    "breeding": "breeding",
    "weaning": "weaning",
    "rumen": "rumen",
    "fatty acid": "fatty acid",
    "fatty acids": "fatty acid",
    "body weight gain": "body weight gain",
    
    # disease-related
    "swine diseases": "swine disease",
    "swine disease": "swine disease",
    "cattle diseases": "cattle disease",
    "cattle disease": "cattle disease",
    
    # molecular / immune / stress
    "gene expressions": "gene expression",
    "gene expression": "gene expression",
    "immune response": "immune response",
    "inflammatory response": "inflammation",
    "inflammation": "inflammation",
    "oxidative stress": "oxidative stress",
    "histopathology": "histopathology",
    "genotype": "genotype",
    "reproduction": "reproduction",
    "pregnancy": "pregnancy"
}

def clean_keyword(term):
    if pd.isna(term):
        return None
    
    term = str(term).strip().lower()
    
    if term in ["", "nan", "none"]:
        return None
    
    # Replace special hyphens
    term = term.replace("–", "-").replace("—", "-")
    
    # Remove unnecessary punctuation but keep spaces, hyphens, and dots
    term = re.sub(r"[^a-z0-9\s\-.]", " ", term)
    term = re.sub(r"\s+", " ", term).strip()
    term = term.strip(" .")
    
    # Synonym normalization
    term = keyword_synonym_map.get(term, term)
    
    # Stopword filtering after normalization
    if term in keyword_stopwords:
        return None
    
    if len(term) <= 1:
        return None
    
    return term


def split_keywords(keyword_text):
    if pd.isna(keyword_text) or str(keyword_text).strip() == "":
        return []
    
    keyword_text = str(keyword_text)
    
    # Scopus keywords are usually separated by semicolon or vertical bar
    raw_terms = re.split(r";|\|", keyword_text)
    
    cleaned_terms = []
    for term in raw_terms:
        cleaned = clean_keyword(term)
        if cleaned is not None:
            cleaned_terms.append(cleaned)
    
    # Remove duplicates within the same article
    cleaned_terms = list(dict.fromkeys(cleaned_terms))
    
    return cleaned_terms

# Cell 25. Keyword long-format 데이터 만들기

In [ ]:
# =========================
# Cell 25. Build keyword long-format dataframe
# =========================

keyword_records = []

for idx, row in df_clean.iterrows():
    year = row["Year"]
    journal = row["Journal"]
    title = row["Title"]
    
    # Author keywords
    author_keywords = []
    if author_kw_col in df_clean.columns:
        author_keywords = split_keywords(row[author_kw_col])
    
    for kw in author_keywords:
        keyword_records.append({
            "Record_ID": idx,
            "Year": year,
            "Journal": journal,
            "Title": title,
            "Keyword": kw,
            "Keyword_type": "Author keyword"
        })
    
    # Indexed keywords
    indexed_keywords = []
    if index_kw_col in df_clean.columns:
        indexed_keywords = split_keywords(row[index_kw_col])
    
    for kw in indexed_keywords:
        keyword_records.append({
            "Record_ID": idx,
            "Year": year,
            "Journal": journal,
            "Title": title,
            "Keyword": kw,
            "Keyword_type": "Indexed keyword"
        })

keyword_long = pd.DataFrame(keyword_records)

# Remove duplicated keyword from same article across author/indexed keyword
keyword_long = keyword_long.drop_duplicates(
    subset=["Record_ID", "Keyword"]
).reset_index(drop=True)

print("Number of keyword records:", len(keyword_long))
print("Number of unique keywords:", keyword_long["Keyword"].nunique())

display(keyword_long.head(20))

# Cell 26. Keyword 결측 및 추출 상태 확

In [ ]:
# =========================
# Cell 26. Keyword extraction summary
# =========================

docs_with_keywords = keyword_long["Record_ID"].nunique()
total_docs = len(df_clean)

print("Total documents:", total_docs)
print("Documents with at least one keyword:", docs_with_keywords)
print("Documents without keywords:", total_docs - docs_with_keywords)
print("Keyword coverage (%):", round(docs_with_keywords / total_docs * 100, 2))

top_keywords_overall = (
    keyword_long
    .groupby("Keyword")
    .agg(
        Frequency=("Record_ID", "nunique")
    )
    .reset_index()
    .sort_values("Frequency", ascending=False)
    .reset_index(drop=True)
)

display(top_keywords_overall.head(50))

# Cell 27. 기간 구분 만들기

In [ ]:
# =========================
# Cell 27. Assign time periods
# =========================

def assign_period(year):
    if 2010 <= year <= 2014:
        return "2010–2014"
    elif 2015 <= year <= 2019:
        return "2015–2019"
    elif 2020 <= year <= 2025:
        return "2020–2025"
    else:
        return np.nan

keyword_long["Period"] = keyword_long["Year"].apply(assign_period)

period_check = (
    keyword_long
    .groupby("Period")
    .agg(
        Keyword_records=("Keyword", "count"),
        Documents=("Record_ID", "nunique"),
        Unique_keywords=("Keyword", "nunique")
    )
    .reset_index()
)

display(period_check)

# Cell 28. Table 2 만들기: 기간별 상위 keyword

In [ ]:
# =========================
# Cell 28. Table 2: top keywords by time period
# =========================

top_n_keywords = 20

# Overall top keywords
overall_top = (
    keyword_long
    .groupby("Keyword")
    .agg(Frequency=("Record_ID", "nunique"))
    .reset_index()
    .sort_values("Frequency", ascending=False)
    .head(top_n_keywords)
    .reset_index(drop=True)
)

overall_top["Overall 2010–2025"] = (
    overall_top["Keyword"] + " (" + overall_top["Frequency"].astype(str) + ")"
)

table2 = pd.DataFrame({
    "Rank": range(1, top_n_keywords + 1),
    "Overall 2010–2025": overall_top["Overall 2010–2025"]
})

# Period-specific top keywords
for period in ["2010–2014", "2015–2019", "2020–2025"]:
    period_top = (
        keyword_long[keyword_long["Period"] == period]
        .groupby("Keyword")
        .agg(Frequency=("Record_ID", "nunique"))
        .reset_index()
        .sort_values("Frequency", ascending=False)
        .head(top_n_keywords)
        .reset_index(drop=True)
    )
    
    table2[period] = (
        period_top["Keyword"] + " (" + period_top["Frequency"].astype(str) + ")"
    )

display(table2)

# Cell 29. Table 2 caption

In [ ]:
# =========================
# Cell 29. Table 2 caption
# =========================

table2_caption = (
    "Table 2. Most frequent author and indexed keywords in selected animal science "
    "and veterinary journals according to the overall dataset and three publication periods."
)

print(table2_caption)
display(table2)

# Cell 30. Figure 4에 사용할 주요 keyword 선택

In [ ]:
# =========================
# Cell 30. Select major keywords for temporal trend heatmap
# =========================

selected_keywords_fig4 = [
    "diet supplementation",
    "lactation",
    "milk",
    "microbiome",
    "gene expression",
    "pregnancy",
    "animal welfare",
    "breeding",
    "weaning",
    "dairying",
    "meat",
    "rumen",
    "fatty acid",
    "oxidative stress",
    "inflammation",
    "immune response",
    "histopathology",
    "growth performance",
    "e. coli",
    "food intake"
]

# Check whether selected keywords exist in keyword_long
available_keywords = set(keyword_long["Keyword"].unique())

missing_keywords = [kw for kw in selected_keywords_fig4 if kw not in available_keywords]
existing_keywords = [kw for kw in selected_keywords_fig4 if kw in available_keywords]

print("Selected keywords:", len(selected_keywords_fig4))
print("Existing keywords:", len(existing_keywords))
print("Missing keywords:", missing_keywords)

display(
    keyword_long[keyword_long["Keyword"].isin(existing_keywords)]
    .groupby("Keyword")
    .agg(Frequency=("Record_ID", "nunique"))
    .reindex(existing_keywords)
    .reset_index()
)

# Cell 31. 연도별 normalized keyword frequency 계산

In [ ]:
# =========================
# Cell 31. Calculate annual normalized keyword frequency
# =========================

year_range = list(range(2010, 2026))

# Annual document count
annual_docs = (
    df_clean
    .groupby("Year")
    .size()
    .reindex(year_range)
    .fillna(0)
    .astype(int)
    .rename("Total_documents")
    .reset_index()
)

# Annual keyword count
keyword_year_count = (
    keyword_long[keyword_long["Keyword"].isin(existing_keywords)]
    .groupby(["Keyword", "Year"])
    .agg(Keyword_count=("Record_ID", "nunique"))
    .reset_index()
)

# Complete keyword-year grid
keyword_year_grid = (
    pd.MultiIndex
    .from_product([existing_keywords, year_range], names=["Keyword", "Year"])
    .to_frame(index=False)
)

keyword_year_trend = (
    keyword_year_grid
    .merge(keyword_year_count, on=["Keyword", "Year"], how="left")
    .merge(annual_docs, on="Year", how="left")
)

keyword_year_trend["Keyword_count"] = keyword_year_trend["Keyword_count"].fillna(0).astype(int)

keyword_year_trend["Normalized_frequency"] = (
    keyword_year_trend["Keyword_count"] /
    keyword_year_trend["Total_documents"] * 100
)

keyword_year_trend["Normalized_frequency"] = keyword_year_trend["Normalized_frequency"].fillna(0)

display(keyword_year_trend.head(30))

# Cell 32. Heatmap용 matrix 만들기

In [ ]:
# =========================
# Cell 32. Prepare heatmap matrix
# =========================

heatmap_matrix = (
    keyword_year_trend
    .pivot(index="Keyword", columns="Year", values="Normalized_frequency")
    .reindex(existing_keywords)
)

display(heatmap_matrix)

# Cell 33. Figure 4: normalized keyword frequency heatmap

In [ ]:
# =========================
# Cell 33. Figure 4. Temporal changes in major keywords
# =========================

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# Figure style
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.linewidth"] = 1
mpl.rcParams["xtick.major.width"] = 1
mpl.rcParams["ytick.major.width"] = 1
mpl.rcParams["xtick.direction"] = "out"
mpl.rcParams["ytick.direction"] = "out"

# Custom colormap: DodgerBlue -> HotPink
custom_cmap = LinearSegmentedColormap.from_list(
    "dodgerblue_hotpink",
    ["dodgerblue", "hotpink"]
)

# Color scale limits
vmin = 0
vmax = heatmap_matrix.values.max()

fig, ax = plt.subplots(figsize=(8.5, 6.5))

im = ax.imshow(
    heatmap_matrix.values,
    aspect="auto",
    interpolation="nearest",
    cmap=custom_cmap,
    vmin=vmin,
    vmax=vmax
)

# Axis labels
ax.set_xlabel("Publication year", fontsize=12)
ax.set_ylabel("Keyword", fontsize=12)

# x-axis
ax.set_xticks(np.arange(len(heatmap_matrix.columns)))
ax.set_xticklabels(heatmap_matrix.columns.astype(int), rotation=45, ha="right", fontsize=10)

# y-axis labels
display_labels = []
for kw in heatmap_matrix.index:
    if kw == "e. coli":
        display_labels.append(r"$\it{E. coli}$")
    else:
        display_labels.append(kw.capitalize())

ax.set_yticks(np.arange(len(heatmap_matrix.index)))
ax.set_yticklabels(display_labels, fontsize=10)

# Tick style
ax.tick_params(axis="both", width=1, length=3)

# Full box style
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)

# -------- Colorbar --------
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label("Normalized frequency (%)", fontsize=11)

# colorbar ticks: exactly matched to data range
tick_step = 2
ticks = list(np.arange(vmin, np.floor(vmax) + 1, tick_step))

# ensure the exact maximum value is included
if ticks[-1] != vmax:
    ticks.append(vmax)

cbar.set_ticks(ticks)
cbar.ax.set_yticklabels([f"{t:.1f}" if t == vmax and t % 1 != 0 else f"{int(t)}" if float(t).is_integer() else f"{t:.1f}" for t in ticks])

cbar.ax.tick_params(labelsize=10, width=1, length=3)
cbar.outline.set_linewidth(1)

plt.tight_layout()
plt.show()

# Cell 34. 확인용: 최근 증가 keyword 정렬

In [ ]:
# =========================
# Cell 34. Check keyword changes between early and recent periods
# =========================

early_years = list(range(2010, 2015))
recent_years = list(range(2020, 2026))

keyword_change_summary = pd.DataFrame({
    "Keyword": existing_keywords,
    "Mean_2010_2014": [
        heatmap_matrix.loc[kw, early_years].mean() for kw in existing_keywords
    ],
    "Mean_2020_2025": [
        heatmap_matrix.loc[kw, recent_years].mean() for kw in existing_keywords
    ]
})

keyword_change_summary["Change"] = (
    keyword_change_summary["Mean_2020_2025"] -
    keyword_change_summary["Mean_2010_2014"]
)

keyword_change_summary = keyword_change_summary.sort_values("Change", ascending=False).reset_index(drop=True)

display(keyword_change_summary)

# Cell 35. Keyword co-occurrence network용 keyword frequency 확인

In [ ]:
# =========================
# Cell 35. Keyword frequency for network analysis
# =========================

keyword_freq_for_network = (
    keyword_long
    .groupby("Keyword")
    .agg(Frequency=("Record_ID", "nunique"))
    .reset_index()
    .sort_values("Frequency", ascending=False)
    .reset_index(drop=True)
)

display(keyword_freq_for_network.head(100))

# Cell 36. 논문별 keyword list 만들기

In [ ]:
# =========================
# Cell 36. Build article-level keyword lists
# =========================

doc_keywords = (
    keyword_long
    .groupby("Record_ID")["Keyword"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)

doc_keywords["Number_of_keywords"] = doc_keywords["Keyword"].apply(len)

print("Number of documents with keywords:", len(doc_keywords))
print("Mean number of keywords per document:", round(doc_keywords["Number_of_keywords"].mean(), 2))
print("Median number of keywords per document:", doc_keywords["Number_of_keywords"].median())

display(doc_keywords.head(10))

# Cell 37. Keyword co-occurrence pair 계산

In [ ]:
# =========================
# Cell 37. Calculate keyword co-occurrence pairs
# =========================

from itertools import combinations
from collections import Counter

# Network filtering parameters
top_n_nodes = 35
min_edge_weight = 14

# Select top keywords by document frequency
top_keywords_network = (
    keyword_freq_for_network
    .head(top_n_nodes)["Keyword"]
    .tolist()
)

top_keywords_set = set(top_keywords_network)

pair_counter = Counter()

for kws in doc_keywords["Keyword"]:
    kws_filtered = [kw for kw in kws if kw in top_keywords_set]
    
    if len(kws_filtered) < 2:
        continue
    
    for kw1, kw2 in combinations(sorted(kws_filtered), 2):
        pair_counter[(kw1, kw2)] += 1

edge_df = pd.DataFrame(
    [(kw1, kw2, weight) for (kw1, kw2), weight in pair_counter.items()],
    columns=["Keyword_1", "Keyword_2", "Weight"]
)

edge_df = edge_df[edge_df["Weight"] >= min_edge_weight].copy()
edge_df = edge_df.sort_values("Weight", ascending=False).reset_index(drop=True)

print("Number of edges:", len(edge_df))
display(edge_df.head(50))

# Cell 38. Network 생성 및 community detection

In [ ]:
# =========================
# Cell 38. Build keyword co-occurrence network
# =========================

import networkx as nx

G = nx.Graph()

# Add nodes with frequency
for _, row in keyword_freq_for_network[keyword_freq_for_network["Keyword"].isin(top_keywords_set)].iterrows():
    G.add_node(
        row["Keyword"],
        frequency=row["Frequency"]
    )

# Add edges
for _, row in edge_df.iterrows():
    G.add_edge(
        row["Keyword_1"],
        row["Keyword_2"],
        weight=row["Weight"]
    )

# Remove isolated nodes
isolated_nodes = list(nx.isolates(G))
G.remove_nodes_from(isolated_nodes)

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Removed isolated nodes:", len(isolated_nodes))

# Community detection
communities = list(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"))

community_dict = {}
for i, community in enumerate(communities, 1):
    for node in community:
        community_dict[node] = i

nx.set_node_attributes(G, community_dict, "community")

community_summary = pd.DataFrame({
    "Community": [i + 1 for i in range(len(communities))],
    "Number_of_keywords": [len(c) for c in communities],
    "Representative_keywords": [
        ", ".join(
            keyword_freq_for_network[
                keyword_freq_for_network["Keyword"].isin(list(c))
            ]
            .sort_values("Frequency", ascending=False)
            .head(8)["Keyword"]
            .tolist()
        )
        for c in communities
    ]
})

display(community_summary)

# Cell 39. Figure 4. Keyword co-occurrence network

In [ ]:
# =========================
# Cell 39. Figure 4. Keyword co-occurrence network
# =========================

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

# Figure style
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.linewidth"] = 1
mpl.rcParams["xtick.major.width"] = 1
mpl.rcParams["ytick.major.width"] = 1
mpl.rcParams["xtick.direction"] = "out"
mpl.rcParams["ytick.direction"] = "out"

# ---------------------------------
# 1. Remove invalid / blank keyword nodes
# ---------------------------------
G_plot = G.copy()

invalid_nodes = [
    n for n in G_plot.nodes()
    if pd.isna(n) or str(n).strip() == ""
]

G_plot.remove_nodes_from(invalid_nodes)

print("Removed blank/invalid nodes:", len(invalid_nodes))
print("Nodes in plot:", G_plot.number_of_nodes())
print("Edges in plot:", G_plot.number_of_edges())

# ---------------------------------
# 2. Layout
# ---------------------------------
fig, ax = plt.subplots(figsize=(5, 5))

pos = nx.spring_layout(
    G_plot,
    k=0.45,
    iterations=300,
    seed=42,
    weight="weight"
)

# ---------------------------------
# 3. Node size by frequency
# ---------------------------------
node_freq = np.array([G_plot.nodes[n]["frequency"] for n in G_plot.nodes()])
node_sizes = 80 + (node_freq / node_freq.max()) * 900

# ---------------------------------
# 4. Community colors: sky-blue / pink palette
# ---------------------------------
community_palette = [
    "#87CEFA",  # light sky blue
    "#F4A6C1",  # soft pink
    "#B0E0E6",  # powder blue
    "#FFB6C1",  # light pink
    "#ADD8E6",  # light blue
    "#FFC0CB",  # pink
    "#CDB4DB",  # soft lavender-pink
    "#A2D2FF",  # pastel sky blue
    "#EFC3E6",  # pale pink-lilac
    "#90E0EF"   # soft cyan-blue
]

node_colors = [
    community_palette[(G_plot.nodes[n]["community"] - 1) % len(community_palette)]
    for n in G_plot.nodes()
]

# ---------------------------------
# 5. Edge width by co-occurrence weight
# ---------------------------------
edge_weights = np.array([G_plot[u][v]["weight"] for u, v in G_plot.edges()])
edge_widths = 0.3 + (edge_weights / edge_weights.max()) * 2.2

nx.draw_networkx_edges(
    G_plot,
    pos,
    ax=ax,
    width=edge_widths,
    alpha=0.30,
    edge_color="gray"
)

# ---------------------------------
# 6. Draw nodes
# ---------------------------------
nx.draw_networkx_nodes(
    G_plot,
    pos,
    ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.92,
    linewidths=0.7,
    edgecolors="black"
)

# ---------------------------------
# 7. Label formatting
# ---------------------------------
def format_keyword_label(kw):
    kw = str(kw).strip()
    
    if kw == "":
        return ""
    if kw == "e. coli":
        return r"$\it{E. coli}$"
    
    return kw.capitalize()

# ---------------------------------
# 8. Draw labels
# ---------------------------------
# 방법 1: 모든 노드 라벨 표시
# label_threshold = 0

# 방법 2: 너무 복잡하지 않게 일부만 표시
label_threshold = np.percentile(node_freq, 0)

labels = {
    node: format_keyword_label(node)
    for node in G_plot.nodes()
    if (
        G_plot.nodes[node]["frequency"] >= label_threshold and
        format_keyword_label(node) != ""
    )
}

nx.draw_networkx_labels(
    G_plot,
    pos,
    labels=labels,
    font_size=8,
    font_family="Arial",
    ax=ax
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

# Cell 40. Figure 4에 사용한 네트워크 요약 확인

In [ ]:
# =========================
# Cell 40. Summary of the network used for Figure 4
# =========================

print("Number of nodes:", G_plot.number_of_nodes())
print("Number of edges:", G_plot.number_of_edges())

print("\nNodes:")
print(sorted(list(G_plot.nodes())))

# Cell 41. Community summary 재정리

In [ ]:
# =========================
# Cell 41. Community summary based on G_plot
# =========================

community_summary_plot = []

for community_id in sorted(set(nx.get_node_attributes(G_plot, "community").values())):
    nodes_in_comm = [
        n for n in G_plot.nodes()
        if G_plot.nodes[n]["community"] == community_id
    ]
    
    nodes_sorted = sorted(
        nodes_in_comm,
        key=lambda n: G_plot.nodes[n]["frequency"],
        reverse=True
    )
    
    community_summary_plot.append({
        "Community": community_id,
        "Number_of_keywords": len(nodes_sorted),
        "Representative_keywords": ", ".join(nodes_sorted[:10])
    })

community_summary_plot = pd.DataFrame(community_summary_plot)

display(community_summary_plot)

# Cell 42. Edge weight 상위 50개 확인

In [ ]:
# =========================
# Cell 42. Top keyword co-occurrence edges based on G_plot
# =========================

edge_summary_plot = []

for u, v, d in G_plot.edges(data=True):
    edge_summary_plot.append({
        "Keyword_1": u,
        "Keyword_2": v,
        "Weight": d["weight"]
    })

edge_summary_plot = (
    pd.DataFrame(edge_summary_plot)
    .sort_values("Weight", ascending=False)
    .reset_index(drop=True)
)

display(edge_summary_plot.head(50))

# Cell 43. Centrality 계산

In [ ]:
# =========================
# Cell 43. Centrality analysis based on G_plot - revised
# =========================

G_cent = G_plot.copy()

# Degree centrality: number of direct connections
degree_centrality = nx.degree_centrality(G_cent)

# Betweenness centrality: bridge role between different parts of the network
# Use unweighted version for comparability with common bibliometric network tables
betweenness_centrality = nx.betweenness_centrality(
    G_cent,
    normalized=True
)

# Closeness centrality: topological closeness to other nodes
# Use unweighted version to keep values within a conventional 0–1 range
closeness_centrality = nx.closeness_centrality(G_cent)

# Eigenvector centrality: connection to influential nodes
# Weight can be retained here because co-occurrence strength is meaningful
eigenvector_centrality = nx.eigenvector_centrality_numpy(
    G_cent,
    weight="weight"
)

centrality_plot = pd.DataFrame({
    "Keyword": list(G_cent.nodes()),
    "Frequency": [G_cent.nodes[n]["frequency"] for n in G_cent.nodes()],
    "Community": [G_cent.nodes[n]["community"] for n in G_cent.nodes()],
    "Degree_centrality": [degree_centrality[n] for n in G_cent.nodes()],
    "Closeness_centrality": [closeness_centrality[n] for n in G_cent.nodes()],
    "Betweenness_centrality": [betweenness_centrality[n] for n in G_cent.nodes()],
    "Eigenvector_centrality": [eigenvector_centrality[n] for n in G_cent.nodes()]
})

centrality_plot = (
    centrality_plot
    .sort_values("Degree_centrality", ascending=False)
    .reset_index(drop=True)
)

display(centrality_plot)

# Cell 44. Table 3 생성

In [ ]:
# =========================
# Cell 44. Table 3. Top keywords by centrality measures - revised
# =========================

top_n_centrality = 10  # reference table style; use 15 if needed

def format_keyword_for_table(keyword):
    if keyword == "e. coli":
        return "_E. coli_"
    if keyword == "messenger rna":
        return "messenger RNA"
    return keyword

def format_metric_table(df, metric, n=10):
    temp = (
        df
        .sort_values(metric, ascending=False)
        .head(n)
        .reset_index(drop=True)
    )
    
    keywords = []
    scores = []
    
    for _, row in temp.iterrows():
        keywords.append(format_keyword_for_table(row["Keyword"]))
        scores.append(round(row[metric], 3))
    
    return keywords, scores

deg_kw, deg_score = format_metric_table(centrality_plot, "Degree_centrality", top_n_centrality)
clo_kw, clo_score = format_metric_table(centrality_plot, "Closeness_centrality", top_n_centrality)
bet_kw, bet_score = format_metric_table(centrality_plot, "Betweenness_centrality", top_n_centrality)
eig_kw, eig_score = format_metric_table(centrality_plot, "Eigenvector_centrality", top_n_centrality)

table3_reference_style = pd.DataFrame({
    "Rank": range(1, top_n_centrality + 1),
    "Degree keyword": deg_kw,
    "Degree score": deg_score,
    "Closeness keyword": clo_kw,
    "Closeness score": clo_score,
    "Betweenness keyword": bet_kw,
    "Betweenness score": bet_score,
    "Eigenvector keyword": eig_kw,
    "Eigenvector score": eig_score
})

display(table3_reference_style)

# Cell 45. 해석용 핵심 keyword의 연결 관계 확인

In [ ]:
# =========================
# Cell 45. Major neighbors of selected key nodes
# =========================

key_nodes = [
    "diet supplementation",
    "microbiome",
    "gene expression",
    "lactation",
    "milk",
    "pregnancy",
    "rumen",
    "animal welfare",
    "swine disease",
    "e. coli"
]

for node in key_nodes:
    if node in G_plot.nodes():
        neighbors = []
        
        for nb in G_plot.neighbors(node):
            neighbors.append({
                "Keyword": nb,
                "Weight": G_plot[node][nb]["weight"],
                "Community": G_plot.nodes[nb]["community"]
            })
        
        neighbors_df = (
            pd.DataFrame(neighbors)
            .sort_values("Weight", ascending=False)
            .head(10)
            .reset_index(drop=True)
        )
        
        print("\n==============================")
        print(node)
        print("==============================")
        display(neighbors_df)

# Cell 46. Table 3 caption

In [ ]:
# =========================
# Cell 46. Table 3 caption
# =========================

table3_caption = (
    "Table 3. Centrality scores for primary keywords in the keyword co-occurrence network."
)

print(table3_caption)
display(table3_reference_style)

# Cell 47 수정본. BERTopic용 텍스트 데이터 준비

In [ ]:
# =========================
# Cell 47. Prepare text data for BERTopic
# =========================

# Check available columns
print("Available columns:")
for col in df_clean.columns:
    print("-", col)

# Make analysis dataframe
topic_df = df_clean.copy()

# Check title column
if "Title" not in topic_df.columns:
    raise ValueError("Column 'Title' was not found. Please check Scopus export columns.")

# Find abstract column
abstract_candidates = [col for col in topic_df.columns if "abstract" in col.lower()]

print("\nAbstract candidate columns:", abstract_candidates)

if "Abstract" in topic_df.columns:
    abstract_col = "Abstract"
elif len(abstract_candidates) > 0:
    abstract_col = abstract_candidates[0]
    print("Using abstract column:", abstract_col)
else:
    raise ValueError("No abstract-related column was found. Please check Scopus export columns.")

# Fill missing title/abstract
topic_df["Title"] = topic_df["Title"].fillna("").astype(str)
topic_df[abstract_col] = topic_df[abstract_col].fillna("").astype(str)

# Combine title and abstract
topic_df["Text"] = (
    topic_df["Title"].str.strip() + ". " +
    topic_df[abstract_col].str.strip()
)

# Basic text length filter
topic_df["Text_length"] = topic_df["Text"].str.split().apply(len)

topic_df_topic = (
    topic_df[topic_df["Text_length"] >= 30]
    .copy()
    .reset_index(drop=True)
)

print("\nOriginal documents:", len(topic_df))
print("Documents used for topic modeling:", len(topic_df_topic))
print("Excluded short-text documents:", len(topic_df) - len(topic_df_topic))

display(topic_df_topic[["Title", "Year", "Journal", "Text_length"]].head())

# Cell 48. BERTopic 설치 및 불러오기

In [ ]:
pip install bertopic

In [ ]:
# =========================
# Cell 48-1. Import BERTopic packages
# =========================

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

import pandas as pd
import numpy as np
import re

# Cell 49. BERTopic용 텍스트 전처리

In [ ]:
# =========================
# Cell 49. Basic text cleaning for BERTopic
# =========================

def clean_text_for_topic(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

docs = topic_df_topic["Text"].apply(clean_text_for_topic).tolist()

print("Number of documents:", len(docs))
print("Example document:")
print(docs[0][:1000])

# Cell 50. BERTopic 모델 설정

In [ ]:
# =========================
# Cell 50. Configure BERTopic model
# =========================

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import os

# -------------------------------------------------
# 1. Load embedding model
# -------------------------------------------------
# Store the embedding model inside the project folder if available.
local_model_path = MODEL_DIR / "all-MiniLM-L6-v2"

print("Local model path:")
print(local_model_path)

if local_model_path.exists():
    print("Loading local SentenceTransformer model...")
    embedding_model = SentenceTransformer(str(local_model_path))
else:
    print("Local model was not found.")
    print("Loading SentenceTransformer model from Hugging Face...")
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# -------------------------------------------------
# 2. UMAP model
# -------------------------------------------------
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# -------------------------------------------------
# 3. HDBSCAN model
# -------------------------------------------------
hdbscan_model = HDBSCAN(
    min_cluster_size=60,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# -------------------------------------------------
# 4. Vectorizer model
# -------------------------------------------------
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80
)

# -------------------------------------------------
# 5. BERTopic model
# -------------------------------------------------
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True
)

print("BERTopic model is ready.")

# Cell 51. BERTopic 학습 실행

In [ ]:
# =========================
# Cell 51. Fit BERTopic model
# =========================

import numpy as np
import pandas as pd

# Basic check
print("Number of documents:", len(docs))
print("Number of rows in topic_df_topic:", len(topic_df_topic))

if len(docs) != len(topic_df_topic):
    raise ValueError("The number of documents and rows in topic_df_topic does not match.")

# Fit BERTopic
topics, probabilities = topic_model.fit_transform(docs)

# Add topic assignment to dataframe
topic_df_topic = topic_df_topic.copy()
topic_df_topic["Topic"] = topics

# If probabilities are available, add max probability
if probabilities is not None:
    try:
        topic_df_topic["Topic_probability"] = np.max(probabilities, axis=1)
    except:
        topic_df_topic["Topic_probability"] = probabilities
else:
    topic_df_topic["Topic_probability"] = np.nan

# Summary
n_total = len(topic_df_topic)
n_outlier = (topic_df_topic["Topic"] == -1).sum()
n_topic = topic_df_topic["Topic"].nunique() - (1 if -1 in topic_df_topic["Topic"].unique() else 0)

print("BERTopic fitting completed.")
print("Number of topics excluding outliers:", n_topic)
print("Number of outlier documents:", n_outlier)
print("Outlier ratio:", round(n_outlier / n_total * 100, 2), "%")

display(topic_df_topic[["Title", "Year", "Journal", "Topic", "Topic_probability"]].head())

# Cell 52. Topic 기본 정보 확인

In [ ]:
# =========================
# Cell 52. Topic information
# =========================

topic_info = topic_model.get_topic_info()

print("Topic information:")
display(topic_info)

# Topic count excluding outlier topic -1
topic_info_no_outlier = topic_info[topic_info["Topic"] != -1].copy()

print("Number of valid topics:", len(topic_info_no_outlier))

# Cell 53. Topic representation 노이즈 제거

In [ ]:
# =========================
# Cell 53. Clean BERTopic topic representations
# =========================

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import pandas as pd

# Additional stopwords for topic representation cleaning
custom_stopwords = set(ENGLISH_STOP_WORDS).union({
    # General academic/reporting terms
    "study", "studies", "result", "results", "method", "methods",
    "analysis", "data", "using", "used", "use", "effect", "effects",
    "different", "significant", "significantly", "group", "groups",
    "control", "treated", "treatment", "experiment", "experimental",
    "objective", "aim", "purpose", "background", "conclusion",
    "article", "review", "paper", "research",
    
    # Journal/publisher/copyright noise
    "nature", "america", "rights", "reserved", "copyright",
    "license", "published", "publisher", "springer", "elsevier",
    "wiley", "permission", "agriculture",
    
    # Very generic animal/veterinary terms
    "animal", "animals", "veterinary", "science"
})

clean_vectorizer_model = CountVectorizer(
    stop_words=list(custom_stopwords),
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80
)

# Update only topic representations.
# This does not redo embedding, UMAP, or HDBSCAN clustering.
topic_model.update_topics(
    docs,
    vectorizer_model=clean_vectorizer_model
)

# Check cleaned topic information
topic_info_clean = topic_model.get_topic_info()

print("Cleaned BERTopic topic information")
display(topic_info_clean)

# Representative terms after cleaning
top_n_terms = 15
topic_terms_clean_records = []

for topic_id in topic_info_clean["Topic"]:
    if topic_id == -1:
        continue
    
    terms = topic_model.get_topic(topic_id)
    
    if terms is None:
        continue
    
    top_terms = terms[:top_n_terms]
    
    topic_terms_clean_records.append({
        "Topic": topic_id,
        "Representative_terms": ", ".join([term for term, score in top_terms]),
        "Terms_with_scores": "; ".join([f"{term} ({score:.3f})" for term, score in top_terms])
    })

topic_terms_clean_df = pd.DataFrame(topic_terms_clean_records)

display(topic_terms_clean_df)

# Cell 54. Reduced topic 수 후보 평가

In [ ]:
# =========================
# Cell 54. Evaluate candidate numbers of reduced BERTopic topics from 2 to 40
# =========================

import copy
import gc
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# -------------------------------------------------
# 0. Stopwords
# -------------------------------------------------
if "custom_stopwords" not in globals():
    custom_stopwords = set(ENGLISH_STOP_WORDS).union({
        # General academic/reporting terms
        "study", "studies", "result", "results", "method", "methods",
        "analysis", "data", "using", "used", "use", "effect", "effects",
        "different", "significant", "significantly", "group", "groups",
        "control", "treated", "treatment", "experiment", "experimental",
        "objective", "aim", "purpose", "background", "conclusion",
        "article", "review", "paper", "research",
        
        # Journal/publisher/copyright noise
        "nature", "america", "rights", "reserved", "copyright",
        "license", "published", "publisher", "springer", "elsevier",
        "wiley", "permission", "agriculture",
        
        # Generic terms
        "animal", "animals", "veterinary", "science",
        
        # Statistical/unit noise
        "05", "005", "kg", "mg", "bw"
    })

# -------------------------------------------------
# 1. Tokenize documents for coherence calculation
# -------------------------------------------------
def tokenize_for_coherence(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    
    tokens = [
        t for t in tokens
        if len(t) > 2 and t not in custom_stopwords
    ]
    
    # Add simple bigram tokens to match BERTopic bigram terms
    bigrams = [
        tokens[i] + "_" + tokens[i + 1]
        for i in range(len(tokens) - 1)
    ]
    
    return tokens + bigrams

tokenized_docs = [tokenize_for_coherence(doc) for doc in docs]
tokenized_docs = [tokens for tokens in tokenized_docs if len(tokens) > 0]

dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(tokens) for tokens in tokenized_docs]

print("Number of tokenized documents:", len(tokenized_docs))
print("Dictionary size:", len(dictionary))

# -------------------------------------------------
# 2. Safe vectorizer for topic reduction
# -------------------------------------------------
# min_df must be 1 because reduced topic numbers can be small.
reduction_vectorizer_model = CountVectorizer(
    stop_words=list(custom_stopwords),
    ngram_range=(1, 2),
    min_df=1,
    max_df=1.0
)

# -------------------------------------------------
# 3. Evaluation functions
# -------------------------------------------------
def normalize_topic_term(term):
    term = str(term).lower().strip()
    term = re.sub(r"[^a-z\s]", " ", term)
    term = re.sub(r"\s+", " ", term).strip()
    term = term.replace(" ", "_")
    return term


def calculate_topic_diversity(topic_words):
    all_words = [word for topic in topic_words for word in topic]
    
    if len(all_words) == 0:
        return np.nan
    
    return len(set(all_words)) / len(all_words)


def evaluate_reduced_bertopic(base_model, docs, nr_topics, top_n_words=10):
    model_copy = copy.deepcopy(base_model)
    
    model_copy.update_topics(
        docs,
        vectorizer_model=reduction_vectorizer_model
    )
    
    reduced_result = model_copy.reduce_topics(
        docs,
        nr_topics=nr_topics
    )
    
    if reduced_result is not None:
        model_copy = reduced_result
    
    topic_info_temp = model_copy.get_topic_info()
    valid_topics = topic_info_temp[topic_info_temp["Topic"] != -1]["Topic"].tolist()
    
    topic_words = []
    
    for topic_id in valid_topics:
        terms = model_copy.get_topic(topic_id)
        
        if terms is None:
            continue
        
        words = []
        
        for term, score in terms[:top_n_words]:
            term_norm = normalize_topic_term(term)
            
            if term_norm in dictionary.token2id:
                words.append(term_norm)
        
        if len(words) >= 3:
            topic_words.append(words)
    
    if len(topic_words) > 0:
        coherence_model_cv = CoherenceModel(
            topics=topic_words,
            texts=tokenized_docs,
            dictionary=dictionary,
            coherence="c_v"
        )
        coherence_cv = coherence_model_cv.get_coherence()
        
        coherence_model_npmi = CoherenceModel(
            topics=topic_words,
            texts=tokenized_docs,
            dictionary=dictionary,
            coherence="c_npmi"
        )
        coherence_npmi = coherence_model_npmi.get_coherence()
    else:
        coherence_cv = np.nan
        coherence_npmi = np.nan
    
    topic_diversity = calculate_topic_diversity(topic_words)
    
    valid_topic_counts = topic_info_temp[topic_info_temp["Topic"] != -1]["Count"].values
    
    if len(valid_topic_counts) > 0:
        largest_topic_share = valid_topic_counts.max() / valid_topic_counts.sum() * 100
        smallest_topic_size = valid_topic_counts.min()
        mean_topic_size = valid_topic_counts.mean()
    else:
        largest_topic_share = np.nan
        smallest_topic_size = np.nan
        mean_topic_size = np.nan
    
    if -1 in topic_info_temp["Topic"].values:
        outlier_count = topic_info_temp.loc[
            topic_info_temp["Topic"] == -1,
            "Count"
        ].values[0]
    else:
        outlier_count = 0
    
    outlier_ratio = outlier_count / len(docs) * 100
    
    result = {
        "Target_topics": nr_topics,
        "Valid_topics": len(valid_topics),
        "Coherence_c_v": coherence_cv,
        "Coherence_c_npmi": coherence_npmi,
        "Topic_diversity": topic_diversity,
        "Smallest_topic_size": smallest_topic_size,
        "Mean_topic_size": mean_topic_size,
        "Largest_topic_share_valid_topics_%": largest_topic_share,
        "Outlier_count": outlier_count,
        "Outlier_ratio_%": outlier_ratio
    }
    
    del model_copy
    gc.collect()
    
    return result

# -------------------------------------------------
# 4. Evaluate candidate topic numbers from 2 to 40
# -------------------------------------------------
candidate_topic_numbers = list(range(2, 41))

evaluation_records = []

for n in candidate_topic_numbers:
    print(f"Evaluating nr_topics = {n} ...")
    
    result = evaluate_reduced_bertopic(
        base_model=topic_model,
        docs=docs,
        nr_topics=n,
        top_n_words=10
    )
    
    evaluation_records.append(result)

topic_reduction_eval = pd.DataFrame(evaluation_records)

display(topic_reduction_eval)

# Cell 55. Topic number 평가 결과 그림

In [ ]:
# =========================
# Cell 55. Plot topic reduction evaluation results
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

# -------------------------------------------------
# 1. Rebuild topic_reduction_eval if it does not exist
# -------------------------------------------------
if "topic_reduction_eval" not in globals():
    topic_reduction_eval = pd.DataFrame({
        "Target_topics": list(range(2, 41)),
        "Valid_topics": [
            1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
            11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
            21, 22, 23, 24, 25, 26, 27, 28, 29, 30,
            31, 32, 33, 34, 35, 36, 37, 38, 39
        ],
        "Coherence_c_v": [
            0.780146, 0.849428, 0.804079, 0.778837, 0.764575,
            0.743372, 0.766043, 0.768768, 0.772862, 0.776178,
            0.788523, 0.799821, 0.791423, 0.791832, 0.786223,
            0.775631, 0.773639, 0.753270, 0.745432, 0.733544,
            0.737385, 0.738692, 0.745372, 0.752728, 0.755732,
            0.756514, 0.759666, 0.763429, 0.762620, 0.769496,
            0.770647, 0.769959, 0.767561, 0.775058, 0.775407,
            0.781628, 0.784621, 0.779629, 0.778645
        ],
        "Topic_diversity": [
            1.000000, 1.000000, 1.000000, 0.971429, 0.977778,
            0.963636, 0.953846, 0.960000, 0.941176, 0.905263,
            0.895238, 0.895652, 0.872000, 0.842105, 0.846154,
            0.849673, 0.820988, 0.824561, 0.834254, 0.836842,
            0.830000, 0.826087, 0.825472, 0.815315, 0.822511,
            0.825726, 0.812749, 0.816092, 0.811111, 0.810714,
            0.799308, 0.801347, 0.798046, 0.787975, 0.782875,
            0.784024, 0.778736, 0.776536, 0.773842
        ]
    })

# -------------------------------------------------
# 2. Basic figure settings
# -------------------------------------------------
plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.linewidth"] = 1.0

color_cv = "#E4AAD9"
color_div = "#6D6D6D"

x = topic_reduction_eval["Target_topics"].to_numpy()
y_cv = topic_reduction_eval["Coherence_c_v"].to_numpy()
y_div = topic_reduction_eval["Topic_diversity"].to_numpy()

# -------------------------------------------------
# 3. Function for tick-aligned y-axis limits
# -------------------------------------------------
def get_tick_aligned_limits(values, step):
    lower = np.floor(np.nanmin(values) / step) * step
    upper = np.ceil(np.nanmax(values) / step) * step
    ticks = np.arange(lower, upper + step / 2, step)
    return lower, upper, ticks

y1_min, y1_max, y1_ticks = get_tick_aligned_limits(y_cv, step=0.02)
y2_min, y2_max, y2_ticks = get_tick_aligned_limits(y_div, step=0.05)

# -------------------------------------------------
# 4. Create figure
# -------------------------------------------------
fig, ax1 = plt.subplots(figsize=(5, 4))
ax2 = ax1.twinx()

# Transparent axis patches
ax1.patch.set_alpha(0)
ax2.patch.set_alpha(0)

# Axes order
ax1.set_zorder(2)
ax2.set_zorder(3)

# -------------------------------------------------
# 5. Plot lines
# -------------------------------------------------
# Coherence c_v
ax1.plot(
    x,
    y_cv,
    color=color_cv,
    marker="o",
    markerfacecolor="none",
    markeredgecolor=color_cv,
    markeredgewidth=1.25,
    linewidth=1.5,
    markersize=7,
    label="Coherence $c_v$",
    zorder=5,
    clip_on=False
)

# Topic diversity
ax2.plot(
    x,
    y_div,
    color=color_div,
    marker="o",
    markerfacecolor="none",
    markeredgecolor=color_div,
    markeredgewidth=1.25,
    linewidth=1.5,
    markersize=7,
    label="Topic diversity",
    zorder=6,
    clip_on=False
)

# -------------------------------------------------
# 6. Axis settings
# -------------------------------------------------
# x-axis: exact range, no margin
ax1.set_xlim(2, 40)
ax1.set_xticks(np.arange(2, 41, 2))
ax1.margins(x=0)

# y-axis
ax1.set_ylim(0.72, 0.86)
ax1.set_yticks(np.arange(0.72, 0.861, 0.02))
ax1.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
ax2.margins(y=0)

ax2.set_ylim(0.75, 1.00)
ax2.set_yticks(np.arange(0.75, 1.001, 0.05))
ax2.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
ax2.margins(y=0)

ax1.set_xlabel("Number of reduced topics", fontsize=13)
ax1.set_ylabel("Coherence $c_v$", fontsize=13)
ax2.set_ylabel("Topic diversity", fontsize=13)

ax1.tick_params(axis="both", direction="out", width=1, length=4, labelsize=11)
ax2.tick_params(axis="y", direction="out", width=1, length=4, labelsize=11)

# -------------------------------------------------
# 7. Put spines behind the data
# -------------------------------------------------
for spine in ax1.spines.values():
    spine.set_linewidth(1.0)
    spine.set_zorder(0)

for spine in ax2.spines.values():
    spine.set_linewidth(1.0)
    spine.set_zorder(0)

# -------------------------------------------------
# 8. Legend
# -------------------------------------------------
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines_1 + lines_2,
    labels_1 + labels_2,
    frameon=False,
    fontsize=11,
    loc="upper right",
    ncol=1,
    handlelength=2
)

plt.tight_layout()
plt.show()

# Cell 56. 최종 topic 수 선택 후 축소

In [ ]:
# =========================
# Cell 56. Reduce BERTopic topics to the final number of topics
# =========================

import copy
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

# -------------------------------------------------
# 1. Define custom stopwords
# -------------------------------------------------
# This block prevents NameError when the kernel was restarted
# or when Cell 53 was not run in the current session.

custom_stopwords = set(ENGLISH_STOP_WORDS).union({
    # General academic/reporting terms
    "study", "studies", "result", "results", "method", "methods",
    "analysis", "data", "using", "used", "use", "effect", "effects",
    "different", "significant", "significantly", "group", "groups",
    "control", "treated", "treatment", "experiment", "experimental",
    "objective", "aim", "purpose", "background", "conclusion",
    "article", "review", "paper", "research",
    
    # Journal/publisher/copyright noise
    "nature", "america", "rights", "reserved", "copyright",
    "license", "published", "publisher", "springer", "elsevier",
    "wiley", "permission", "agriculture",
    
    # Very generic animal/veterinary terms
    "animal", "animals", "veterinary", "science",
    
    # Statistical/unit noise
    "05", "005", "kg", "mg", "bw"
})

# -------------------------------------------------
# 2. Final topic reduction setting
# -------------------------------------------------
# Based on coherence, diversity, topic-size balance, and interpretability,
# nr_topics = 15 was selected.
final_nr_topics = 15

# Safe vectorizer for final topic reduction
# min_df=1 is needed because BERTopic reduction uses topic-level documents.
final_vectorizer_model = CountVectorizer(
    stop_words=list(custom_stopwords),
    ngram_range=(1, 2),
    min_df=1,
    max_df=1.0
)

# -------------------------------------------------
# 3. Copy original BERTopic model and apply final reduction
# -------------------------------------------------
topic_model_final = copy.deepcopy(topic_model)

# Update topic representation using the cleaned vectorizer
topic_model_final.update_topics(
    docs,
    vectorizer_model=final_vectorizer_model
)

# Reduce topics to final number
reduced_result = topic_model_final.reduce_topics(
    docs,
    nr_topics=final_nr_topics
)

if reduced_result is not None:
    topic_model_final = reduced_result

# -------------------------------------------------
# 4. Add final topic assignments to dataframe
# -------------------------------------------------
topic_df_topic = topic_df_topic.copy()
topic_df_topic["Topic_final"] = topic_model_final.topics_

# -------------------------------------------------
# 5. Check final topic information
# -------------------------------------------------
topic_info_final = topic_model_final.get_topic_info()

print("Final reduced BERTopic topic information")
print("Selected nr_topics:", final_nr_topics)
print(
    "Number of topics including outlier:",
    len(topic_info_final)
)
print(
    "Number of valid topics excluding outlier:",
    len(topic_info_final[topic_info_final["Topic"] != -1])
)

display(topic_info_final)

# Cell 57. 최종 topic 대표단어 정리

In [ ]:
# =========================
# Cell 57. Representative terms for final BERTopic topics
# =========================

import pandas as pd

top_n_terms = 15
topic_terms_final_records = []

for topic_id in topic_info_final["Topic"]:
    if topic_id == -1:
        continue
    
    terms = topic_model_final.get_topic(topic_id)
    
    if terms is None:
        continue
    
    top_terms = terms[:top_n_terms]
    
    topic_terms_final_records.append({
        "Topic": topic_id,
        "Representative_terms": ", ".join([term for term, score in top_terms]),
        "Terms_with_scores": "; ".join([f"{term} ({score:.3f})" for term, score in top_terms])
    })

topic_terms_final_df = pd.DataFrame(topic_terms_final_records)

display(topic_terms_final_df)

# Cell 58. Table 4 후보 생성

In [ ]:
# =========================
# Cell 58. Build Table 4 candidate from final BERTopic topics
# =========================

topic_summary_records = []

journal_col = "Journal" if "Journal" in topic_df_topic.columns else "Source title"

for topic_id in sorted(topic_df_topic["Topic_final"].unique()):
    if topic_id == -1:
        continue
    
    temp = topic_df_topic[topic_df_topic["Topic_final"] == topic_id].copy()
    
    rep_terms = topic_terms_final_df.loc[
        topic_terms_final_df["Topic"] == topic_id,
        "Representative_terms"
    ].values[0]
    
    topic_name = topic_info_final.loc[
        topic_info_final["Topic"] == topic_id,
        "Name"
    ].values[0]
    
    major_journals = (
        temp[journal_col]
        .value_counts()
        .head(3)
        .index
        .tolist()
    )
    
    topic_summary_records.append({
        "Topic": topic_id,
        "BERTopic name": topic_name,
        "Provisional topic label": "",
        "Representative terms": rep_terms,
        "No. of documents": len(temp),
        "Share of documents (%)": round(len(temp) / len(topic_df_topic) * 100, 2),
        "Main journals": "; ".join(major_journals),
        "Year range": f"{int(temp['Year'].min())}–{int(temp['Year'].max())}"
    })

table4_candidate = (
    pd.DataFrame(topic_summary_records)
    .sort_values("No. of documents", ascending=False)
    .reset_index(drop=True)
)

display(table4_candidate)

In [ ]:
# =========================
# Cell 58-1. Add provisional topic labels to Table 4
# =========================

topic_label_map = {
    0: "Feed supplementation and growth performance",
    1: "Ruminant nutrition and dairy production",
    2: "Viral infection, replication, and vaccination",
    3: "Bacterial infection and antimicrobial resistance",
    4: "Genetic traits, breeding, and genomic selection",
    5: "Sow, piglet, and swine welfare management",
    6: "Livestock welfare and production systems",
    7: "Reproductive physiology and assisted reproduction",
    8: "Parasitic infection and epidemiology",
    9: "Pain, anesthesia, and surgical care",
    10: "Muscle, adipose tissue, and gene regulation",
    11: "Fish disease and aquaculture health",
    12: "Toxicity, oxidative stress, and antioxidant response",
    13: "Prion disease and transmissible spongiform encephalopathies"
}

table4_candidate["Provisional topic label"] = table4_candidate["Topic"].map(topic_label_map)

display(table4_candidate)

In [ ]:
# =========================
# Cell 58-2. Finalize Table 4 object for Table 4 and Figure 6
# =========================

import pandas as pd

# -------------------------------------------------
# 1. Select source table
# -------------------------------------------------
if "table4_final" in globals():
    table4_source = table4_final.copy()
elif "table4_candidate" in globals():
    table4_source = table4_candidate.copy()
else:
    raise NameError("Neither table4_final nor table4_candidate is defined.")

table4_final = table4_source.copy()

# -------------------------------------------------
# 2. Convert BERTopic internal topic number to manuscript topic number
# -------------------------------------------------
# If Topic starts from 0, convert to 1–14.
# If Topic already starts from 1, keep it.
if table4_final["Topic"].min() == 0:
    table4_final["Topic"] = table4_final["Topic"] + 1

# -------------------------------------------------
# 3. Standardize topic label column
# -------------------------------------------------
if "Topic label" not in table4_final.columns:
    if "Provisional topic label" in table4_final.columns:
        table4_final = table4_final.rename(
            columns={"Provisional topic label": "Topic label"}
        )
    else:
        raise KeyError("Topic label column was not found.")

# -------------------------------------------------
# 4. Standardize representative terms exactly as final Table 4
# -------------------------------------------------
final_terms_map = {
    1: "diet, dietary, diets, feed, growth, intestinal, performance, fed, increased, pigs",
    2: "rumen, milk, cows, diet, intake, production, fed, feed, increased, dairy",
    3: "virus, infection, viral, cells, PRRSV, disease, infected, replication, vaccine, influenza",
    4: "infection, Salmonella, isolates, E. coli, mastitis, resistance, strains, Staphylococcus aureus, cells, virulence",
    5: "genetic, traits, breeding, genomic, selection, breeds, population, breed, milk, cattle",
    6: "sows, tail, piglets, pigs, litter, behaviour, welfare, weaning, piglet, sow",
    7: "welfare, systems, livestock, dairy, farm, cows, farms, production, cow, farming",
    8: "sperm, semen, pregnancy, oocytes, fertility, spermatozoa, embryo, embryos, estrus, cells",
    9: "infection, sheep, parasite, Haemonchus contortus, prevalence, resistance, Toxoplasma gondii, immune, cattle, infected",
    10: "pain, dogs, anesthesia, castration, surgical, wound, surgery, mice, rats, analgesia",
    11: "expression, muscle, miR, differentiation, genes, adipose, miRNAs, cells, tissue, fat",
    12: "fish, infection, salmon, PRV, disease, virus, viral, trout, Atlantic salmon",
    13: "rats, induced, toxicity, extract, oxidative, levels, antioxidant, liver, oxidative stress, protective",
    14: "scrapie, prion, BSE, PRNP, CWD, prion protein, PrP, disease, goats, deer"
}

table4_final["Representative terms"] = table4_final["Topic"].map(final_terms_map)

# -------------------------------------------------
# 5. Rename share column if needed
# -------------------------------------------------
if "Share of documents (%)" in table4_final.columns:
    table4_final = table4_final.rename(
        columns={"Share of documents (%)": "Share of total documents (%)"}
    )

# -------------------------------------------------
# 6. Keep manuscript-friendly columns
# -------------------------------------------------
keep_cols = [
    "Topic",
    "Topic label",
    "Representative terms",
    "No. of documents",
    "Share of total documents (%)",
    "Main journals",
    "Year range"
]

table4_final = table4_final[keep_cols].copy()

display(table4_final)

# Cell 59. Figure 6용 데이터 만들기

In [ ]:
# =========================
# Cell 59. Prepare annual topic-share data for Figure 6
# =========================

import pandas as pd
import numpy as np

# -------------------------------------------------
# 1. Check required dataframe
# -------------------------------------------------
if "topic_df_topic" not in globals():
    raise NameError("topic_df_topic is not defined. Run previous BERTopic cells first.")

if "Topic_final" not in topic_df_topic.columns:
    raise KeyError("Topic_final column was not found. Run Cell 56 first.")

if "table4_final" not in globals():
    print("table4_final was not found. Topic labels will be generated from topic number.")
    table4_for_label = None
else:
    table4_for_label = table4_final.copy()

# -------------------------------------------------
# 2. Exclude outlier topic
# -------------------------------------------------
topic_time_df = topic_df_topic[
    topic_df_topic["Topic_final"] != -1
].copy()

topic_time_df["Year"] = topic_time_df["Year"].astype(int)

# -------------------------------------------------
# 3. Create manuscript topic numbers
# -------------------------------------------------
# BERTopic internal topics are 0–13.
# For manuscript presentation, convert them to Topic 1–14.
topic_time_df["Manuscript_topic"] = topic_time_df["Topic_final"] + 1

# -------------------------------------------------
# 4. Count documents by year and topic
# -------------------------------------------------
topic_year_count = (
    topic_time_df
    .groupby(["Year", "Manuscript_topic"])
    .size()
    .reset_index(name="Documents")
)

# -------------------------------------------------
# 5. Calculate annual topic share (%)
# -------------------------------------------------
annual_total = (
    topic_time_df
    .groupby("Year")
    .size()
    .reset_index(name="Annual_documents")
)

topic_year_share = topic_year_count.merge(
    annual_total,
    on="Year",
    how="left"
)

topic_year_share["Annual_share_%"] = (
    topic_year_share["Documents"] /
    topic_year_share["Annual_documents"] *
    100
)

# -------------------------------------------------
# 6. Pivot table for heatmap
# -------------------------------------------------
topic_year_pivot = (
    topic_year_share
    .pivot(
        index="Manuscript_topic",
        columns="Year",
        values="Annual_share_%"
    )
    .fillna(0)
)

# Ensure topic order 1–14
topic_year_pivot = topic_year_pivot.sort_index()

print("Topic-year share table for Figure 6:")
display(topic_year_pivot.round(2))

# Cell 60. Figure 6 heatmap 그리기

In [ ]:
# =========================
# Cell 60. Figure 6. Temporal evolution heatmap of BERTopic topics
# =========================

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
import textwrap

# -------------------------------------------------
# 1. Figure settings
# -------------------------------------------------
plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.linewidth"] = 1.0

axis_label_size = 13
tick_label_size = 11
term_label_size = 8.3

heatmap_cmap = plt.cm.YlGnBu

# -------------------------------------------------
# 2. Check required data
# -------------------------------------------------
if "topic_year_pivot" not in globals():
    raise NameError("topic_year_pivot is not defined. Run Cell 59 first.")

if "table4_final" not in globals():
    raise NameError("table4_final is not defined. Run Cell 58-2 first.")

# -------------------------------------------------
# 3. Prepare heatmap data
# -------------------------------------------------
heatmap_data = topic_year_pivot.copy()

years = heatmap_data.columns.tolist()
topics = heatmap_data.index.tolist()

# -------------------------------------------------
# 4. Representative terms from final Table 4
# -------------------------------------------------
topic_term_map = dict(
    zip(
        table4_final["Topic"],
        table4_final["Representative terms"]
    )
)

y_labels = [f"Topic {topic}" for topic in topics]

right_terms = []
for topic in topics:
    terms = topic_term_map.get(topic, "")
    terms_wrapped = textwrap.fill(str(terms), width=62)
    right_terms.append(terms_wrapped)

# -------------------------------------------------
# 5. Create figure layout
# -------------------------------------------------
fig = plt.figure(figsize=(12.6, 5.8))

gs = GridSpec(
    nrows=1,
    ncols=3,
    width_ratios=[4.7, 4.2, 0.18],
    wspace=0.03
)

ax = fig.add_subplot(gs[0, 0])
ax_text = fig.add_subplot(gs[0, 1])
cax = fig.add_subplot(gs[0, 2])

# -------------------------------------------------
# 6. Draw heatmap
# -------------------------------------------------
im = ax.imshow(
    heatmap_data.values,
    aspect="auto",
    cmap=heatmap_cmap,
    interpolation="nearest",
    vmin=0,
    vmax=np.nanmax(heatmap_data.values)
)

# -------------------------------------------------
# 7. Heatmap axis settings
# -------------------------------------------------
ax.set_xticks(np.arange(len(years)))
ax.set_xticklabels(
    years,
    rotation=90,
    ha="center",
    va="top",
    fontsize=tick_label_size
)

ax.set_yticks(np.arange(len(topics)))
ax.set_yticklabels(
    y_labels,
    fontsize=tick_label_size
)

ax.set_xlabel("Year", fontsize=axis_label_size)
ax.set_ylabel("BERTopic", fontsize=axis_label_size)

ax.tick_params(
    axis="both",
    direction="out",
    width=1,
    length=4,
    labelsize=tick_label_size
)

ax.xaxis.set_ticks_position("bottom")
ax.xaxis.set_label_position("bottom")

# -------------------------------------------------
# 8. Right-side representative terms
# -------------------------------------------------
ax_text.set_xlim(0, 1)
ax_text.set_ylim(len(topics) - 0.5, -0.5)
ax_text.axis("off")

for i, topic in enumerate(topics):
    ax_text.text(
        0.0,
        i,
        right_terms[i],
        fontsize=term_label_size,
        va="center",
        ha="left"
    )

# -------------------------------------------------
# 9. Colorbar
# -------------------------------------------------
cbar = fig.colorbar(im, cax=cax)

cbar.set_label(
    "Annual topic share (%)",
    fontsize=axis_label_size
)

cbar.ax.tick_params(
    labelsize=tick_label_size,
    width=1,
    length=4,
    direction="out"
)

# -------------------------------------------------
# 10. Spine style
# -------------------------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.0)

for spine in cax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.show()